# 05 · 完整流程:一次跑遍全部

这本把 **05_Shanghai Power to Form Students** 的整套流程**串成一条**,一次执行完:
数据 → 映射 → 只调高度天际线 → 权力算子四体制 → 3D 量体/OBJ → 生成 06 互动 html →(可选)换地方跑遍所有街道。

**所有图都会存到 `out/<slug>/Step_05/<时间戳>/`**;最后一格另把互动 html 出到 `06_AI Render Students/out/<slug>/canvas.html`。

> 原理拆解请看引擎代码(`engine/`);算子/体制定义在 `regimes.yaml` / `power_scenarios.yaml`。
> 这本只做一件事:**把整条流程全跑一遍、把产物全落盘。**

## 怎么执行
- 菜单 **Run → Run All Cells** 从头跑到尾;或点一格按 **Shift+Enter** 逐格跑。
- 换站点:改 `config.py` 的 `SLUG`,或在下面第一格后加一行 `config.SLUG = "waitan"`。
- 跑遍所有已缓存街道:把下面的 `RUN_ALL_SITES` 改成 `True`(较久)。

In [ ]:
# 让 notebook 找到 engine 里的代码(这格不用改,直接执行)
import sys, os
sys.path.insert(0, os.path.abspath("engine"))
# 清掉旧的引擎模块缓存:改了 config/engine 后,重跑本格即刻生效(免重启内核)
for _m in [k for k in list(sys.modules) if k in ("config", "common", "operators", "measure")
           or k == "plots" or k.startswith("plots.") or k == "prints" or k.startswith("prints.")]:
    del sys.modules[_m]
import config, common as C, plots, prints
import operators as ops, measure as M
prints.ready()

# True = 最后一格把整套流程跑遍所有已缓存街道(较久);False = 只跑当前 SLUG。
# 批量跑全部:设环境变量 RUN_ALL_SITES=1 再执行(学生默认 False,不改这行也能全跑)。
RUN_ALL_SITES = os.environ.get("RUN_ALL_SITES", "0") == "1"
STAMP = plots.capture("05")     # 本册每张图都存到 out/<slug>/Step_05/<时间戳>/
print("输出时间戳:", STAMP, "→", "out/%s/Step_05/%s/" % (config.SLUG, STAMP))
print("RUN_ALL_SITES =", RUN_ALL_SITES)

## 阶段 A · 数据 → 映射(合并 01 + 02)
读数据(有数据集就现建、否则用随包缓存)→ 看多源覆盖率 → 贴角色 → 反事实试改一条查表 → 画权力地图。

In [ ]:
df, source = C.build_or_load(config.SLUG)     # 有数据集就现建、否则用随包缓存(分支在 build_or_load 里)
prints.prepared(df, source)
prints.coverage(df)                           # 多源覆盖率
df = C.assign_all(df)                          # 贴角色 stakeholder(读 shanghai_lookup.yaml)
prints.stakeholders(df)
res = C.counterfactual(df, frm="developer", to="state")   # 反事实(不动文件、内存里试):看分布怎么变
prints.counterfactual(res)
plots.data_overview(df)                        # 原料:footprint + 实测高度分布
plots.power_map(df)                            # 权力地图:按角色著色

## 阶段 B · 只调高度,看天际线(03 主册)
锁死 footprint 与角色,只放开高度。套 4 个情景横向比——差异完全来自权力配置。

In [ ]:
try:
    plots.satellite_figureground(df)           # Step0 卫星底(需联网;失败自动跳过)
except Exception as e:
    prints.skipped("卫星", e)
scen = C.load_scenarios()
prints.scenarios(scen)
plots.policy_heatmap(scen)                      # 高度政策热力图:红=拔高 / 蓝=压低
SCEN_NAMES = ["current", "developer_led", "community_led", "state_eco"]
heights = {n: (C.scenario_heights(df, scen[n]) if n != "current" else df["height_m"]) for n in SCEN_NAMES}
plots.skyline_panels(df, heights)              # 4 连幅天际线,同色阶可横比
plots.metrics(df, heights)                      # 总GFA / 平均高 / 最高 / 各角色平均高

## 阶段 C · 权力算子 → 四种形态(04 进阶)
权力现在能改**形态语法**:拆板成塔 / 细粒自建 / 向权力重心收拢 / 放大。再把动词串成 4 种体制横比 + 量化指纹。

In [ ]:
recs = C.to_recs(df)
prints.vocab(recs)
b = ops.freeze(recs, who=["state"])            # 先锁定政府/公共
plots.operator_demo(b, ops.split_to_towers(b, target=["resident", "developer"], above_m2=1500, k=3),
                    "拆板成塔 split_to_towers(k=3)")
plots.operator_demo(b, ops.infill(b, target=["resident", "developer"], cell_m2=120),
                    "居民自建 infill(cell≈120㎡)", color="h")
plots.operator_demo(b, ops.concentrate(b), "中央集权 concentrate(向权力重心收拢)", color="h")
plots.operator_demo(recs, ops.scale(recs, target=["state"], factor=2.0),
                    "放大算子 scale(state ×2,依几何中心)")   # 用未冻结的 recs 才看得到 state 变化

regs = ops.load_regimes()
prints.regimes(regs)
after = {n: ops.apply_regime(recs, regs[n]) for n in regs}    # 4 种权力体制各跑一遍
labels = {n: regs[n]["label"] for n in regs}
plots.regime_compare(recs, after, labels=labels)             # 同色阶横比:四种权力 → 四种形态
rows, _ = M.compare(recs, after, config.SLUG)
plots.fingerprint_bars(rows, labels={"current": "现状", **labels})   # 形态指纹柱状图
prints.fingerprints(rows, {"current": "现状", **labels})

## 阶段 D · 3D 量体 + OBJ
把真实 footprint 按高度挤成 3D。OBJ 写到 `out/<slug>/`,并复制一份进本次 `Step_05/<时间戳>/`。

In [ ]:
import shutil
plots.city_3d(df, top=120)                                   # 现状 3D(取占地最大 120 栋,画得快)
dev = df.copy(); dev["height_m"] = heights["developer_led"].values
plots.city_3d(dev, top=120)                                  # developer_led 情景的 3D 天际线
objp, nv, nf = C.export_obj(df, config.SLUG)                 # 全部楼挤成 OBJ → out/<slug>/
dst = C.OUT / config.SLUG / "Step_05" / STAMP / objp.name
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(objp, dst)                                   # 也放一份到本次 Step_05/<时间戳>/
prints.obj_written(objp, nv, nf)
print("OBJ 也复制到:", dst.relative_to(C.ROOT))

## 阶段 E(可选)· 换地方:跑遍所有已缓存街道(05)
把顶部 `RUN_ALL_SITES` 改成 `True` 才会跑。每个街道各自输出到自己的 `out/<slug>/Step_05/<时间戳>/`。
这正是进阶册的点题:**权力的几何逻辑对基底不变,但结果在乎基底。**

In [ ]:
def run_site(slug):
    """对一个街道跑一遍精简全流程(权力地图 + 天际线 + 四体制 + OBJ)。"""
    plots.capture("05")                                      # 每站各开一个时间戳夹、序号归零
    d = C.assign_all(C.current_buildings(slug))
    plots.power_map(d)
    sc = C.load_scenarios()
    hs = {n: (C.scenario_heights(d, sc[n]) if n != "current" else d["height_m"])
          for n in ["current", "developer_led", "community_led", "state_eco"]}
    plots.skyline_panels(d, hs)
    r = C.to_recs(d); rg = ops.load_regimes()
    af = {n: ops.apply_regime(r, rg[n]) for n in rg}
    plots.regime_compare(r, af, labels={n: rg[n]["label"] for n in rg})
    C.export_obj(d, slug)
    print("  完成", slug)

if RUN_ALL_SITES:
    cached = sorted(p.name for p in C.DATA.iterdir() if (p / "buildings.parquet").exists())
    print("已缓存街道:", cached)
    for s in cached:
        config.SLUG = s
        print("==", s, "==")
        run_site(s)
    print("全部完成。每站产物在 out/<slug>/Step_05/<时间戳>/")
else:
    print("RUN_ALL_SITES = False → 只跑了当前站点。要跑遍所有街道,把顶部改成 True 再 Run All。")

## 阶段 F · 生成 06 的 html(canvas.html + report.html)并回链 05 结果
把本站点体块交给隔壁 `06_AI Render Students`,一次产出**两份自含 html**:
- **`06/out/<slug>/canvas.html`** — 互动画布(Three.js 内联):卫星地面 + 各权力体制体块,拖拽找角度、导出 massing / ControlNet 条件图。
- **`06/out/<slug>/report.html`** — AI 渲染报告:每体制 结构层(体块) ↔ 表面层(AI 渲染) ↔ 提示词三层 + 过渡动画,聚合成一页。

并在本次 **`out/<slug>/Step_05/<时间戳>/index.html`** 写一张「05 结果页」:列出本次跑的图,底部一键跳到上面两份 06 html。三者互链:

```
05 Step_05/<时间戳>/index.html ──▶ 06 canvas.html ⇄ 06 report.html
```

> 这是本工作坊的 html 看图入口(原 05 web 报告已移除)。首次需装 06 依赖:`pip install -r "06_AI Render Students/engine/requirements.txt"`。
> `RUN_ALL_SITES=True` 时,对所有已缓存街道各出一套(canvas + report + 结果页)。


In [ ]:
# 把本站体块接到 06:出 canvas.html + report.html,并在 05 结果目录写「前向链接页」
# 链路:05 Step_05/<时间戳>/index.html  →  06 canvas.html  ⇄  06 report.html
# 06 唯一入口 = run.py(设定都在 06 的 config.yaml;程式在 06/engine)
import subprocess, sys
R06 = C.ROOT.parent / "06_AI Render Students"
sites = ([p.name for p in C.DATA.iterdir() if (p / "buildings.parquet").exists()]
         if RUN_ALL_SITES else [config.SLUG])
for s in sites:
    print("== 06 产出:", s, "==")
    # run.py <slug> = 该站 canvas.html + report.html(两者互链)
    subprocess.run([sys.executable, "run.py", s], cwd=str(R06))
    # 05 结果页:取本站最新 Step_05/<时间戳>/,写 index.html 前向链到 06 canvas / report
    step_root = C.OUT / s / "Step_05"
    stamps = sorted(p for p in step_root.iterdir() if p.is_dir()) if step_root.exists() else []
    if stamps:
        subprocess.run([sys.executable, "run.py", "bridge", s, str(stamps[-1])], cwd=str(R06))
        print("  ↪ 05 结果页:", (stamps[-1] / "index.html").relative_to(C.ROOT))
    cv = R06 / "out" / s / "canvas.html"
    print("  ✅ " + str(cv) if cv.exists() else
          "  ⚠ 未生成 —— 请先装 06 依赖:pip install -r \"06_AI Render Students/engine/requirements.txt\"")
print("\n入口:打开 05 结果页 Step_05/<时间戳>/index.html → 一键跳 06 canvas.html / report.html(canvas 与 report 互链)。")

## 小结 & 诚实边界
- 本册把整条流程**串起来一次跑完**、把产物集中到 `out/<slug>/Step_05/<时间戳>/`,并出一份 `06/out/<slug>/canvas.html`。
- 数据关:EULUC 面级优先、danwei 看不见、informal 恒空、AI 高度对极端超高层可能低估。
- 算子/体制是**教学假设、可争论的语言**,不含退线/日照/产权/参与;零 AI 推断。
- 这是 forward 教学练习(改权力 → 看形态),**不是产权认定或规划预测**。